# reFuel to MOTEL unmapped records

This notebook structures the conversion of reFuel technology rows into MOTEL-compatible unmapped records.

## Goals
- Read reFuel input data from Excel.
- Load and inspect the MOTEL schema.
- Map each source row into a normalized record payload.
- Validate required fields and export records for review.

## Output target
- Write records to `dev2/motel-db/records/` as JSON or YAML files.

In [23]:
save_to_db = True

In [1]:
## --- load packages
from pathlib import Path
import pandas as pd
import yaml

# 1) Inputs and setup

## 1.1 Source data (reFuel Excel)

In [2]:
path_refuel = Path(r"E:\Barton\repositories\motel-platform\dev2\data\reFuel_TechDatabase_2026-06-02.xlsx")

refuel_data = pd.read_excel(path_refuel, sheet_name=None)
print(f"sheets found: {list(refuel_data.keys())}")

sheets found: ['README', 'Metadata', 'Predefined', 'Unique values', 'Glossary', 'Carrier', 'Location', 'Currency', 'Process', 'Process ID', 'Technology ID', 'LCA ID', 'Reference ID', 'SolarTech', 'ConvTech', 'StorTech', 'TechMaturity', 'ConvTech(old)', 'Uncertainty', 'Tech Constraints', 'ImportExport Constraints', 'Resource Constraints', 'CO2 Constraints', 'Tech data (long)', 'System setup', 'MOTEL']


## 1.2 Load MOTEL schema

In [4]:
path_schema = r'schema\motel_schema.yaml'

with open(path_schema, "r", encoding="utf-8") as f:
    schema_motel = yaml.safe_load(f)

print(f"Schema loaded successfully")
print(f"Type: {type(schema_motel)}")
if isinstance(schema_motel, dict):
    print(f"Top-level keys ({len(schema_motel)}): {list(schema_motel.keys())[:10]}")

Schema loaded successfully
Type: <class 'dict'>
Top-level keys (13): ['unmapped_record', 'mapped_record', 'technology', 'process', 'source', 'contributor', 'review', 'attribute', 'carrier', 'geographic_scope']


# 2) Create secondary data from Refuel.ch data

## 2.1 Technology and process datasets (done)

source: sheet 'Technology ID'

In [5]:
sheet_name = 'Technology ID'
df_tech_id = refuel_data[sheet_name].copy()

## correct column names
df_tech_id.columns = df_tech_id.loc[1]
df_tech_id = df_tech_id.loc[2:].reset_index(drop=True)

df_tech_id

1,sheet,tech_id,ehubX Tech ID,process_id,process_type,unit_operation,tech_ontology_IRI,description,tech_year,main_sector,main_category,category_spec,tech_type
0,ConvTech,Ammonia_HaberBosch,T_C_Conv_Ammonia_HaberBosch,Ammonia_HaberBosch,Ammonia Synthesis,Haber Bosch unit,,Ammonia synthesis via Haber Bosch process.,2050,Ammonia,Chemical Synthesis,Haber Bosch,Conversion
1,ConvTech,Ammonia_OCGT,T_C_Conv_Ammonia_OCGT,Ammonia_OCGT,Electricity Generation,OCGT,,,2050,Electricity,Gas Turbine,Ammonia fueled,Conversion
2,ConvTech,Amine_Washing_CO2,T_C_Conv_CarbonCapture_Amine,CarbonCapture_Amine,CO2 Capture,Amine washing unit,,CO2 post combustion capture via Amine washing....,"2030, 2040, 2050",Carbon Capture,Point Source Capture,Amine washing,Conversion
3,ConvTech,Waste_Venting_CO2,T_C_Conv_CarbonCapture_CO2Waste_Vent,CarbonCapture_CO2Waste_Vent,CO2 Capture,Ventilator,,CO2 venting from waste-related processes.,2050,Carbon Capture,Ventilation,Waste,Conversion
4,ConvTech,Wood_Venting_CO2,T_C_Conv_CarbonCapture_CO2Wood_Vent,CarbonCapture_CO2Wood_Vent,CO2 Capture,Ventilator,,CO2 venting from wood-related processes.,2050,Carbon Capture,Ventilation,Wood,Conversion
...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,StorTech,CarbonDioxide_Tank,T_S_CarbonDioxide_Tank,NaN,Storage,Tank,,NaN,na,Carbo Dioxide,,,Storage
91,StorTech,CarbonMonoxide_Tank,T_S_CarbonMonoxide_Tank,NaN,Storage,Tank,,NaN,na,Carbon Monoxide,,,Storage
92,StorTech,C8_Hydrocarbons_Tank,T_S_C8_Hydrocarbons_Tank,NaN,Storage,Tank,,NaN,na,Hydrocarbons,,,Storage
93,StorTech,C12_Hydrocarbons_Tank,T_S_C12_Hydrocarbons_Tank,NaN,Storage,Tank,,NaN,na,Hydrocarbons,,,Storage


In [6]:
df_proc_id = refuel_data['Process ID'].copy()
df_proc_id.columns = df_proc_id.loc[0]
df_proc_id = df_proc_id.loc[1:].reset_index(drop=True)

df_proc_id

,Process ID,Process Description,Tech Type,Process Category,Primay Feedstock,Feedstocks,Products,Reference,Assumption
0,Ammonia_HaberBosch,Synthesizing Ammonia from Haber Bosch Process,Conversion,Haber Bosch,NaN,NaN,NaN,NaN,NaN
1,biomass_gasification_to_syngas,Biomass gasification to syngas,Conversion,Gasification,Syngas,"Wood, O2",H2,NaN,NaN
2,CarbonCapture_Amine,CO2 post combustion capture via Amine washing....,Conversion,Carbon Capture,NaN,"CO2, amine",NaN,NaN,NaN
3,CarbonCapture_CO2Waste_Vent,CO₂ venting from waste-related processes.,Conversion,Carbon Capture,NaN,waste,NaN,NaN,NaN
4,CarbonCapture_CO2Wood_Vent,CO₂ venting from wood-related processes.,Conversion,Carbon Capture,NaN,wood,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
80,ThHt_Vent,Ventilation heat loss at high-temperature heat...,Conversion,Heat loss,NaN,heat,NaN,NaN,NaN
81,ThLt_Vent,Ventilation heat loss at low-temperature heat ...,Conversion,Heat loss,NaN,heat,NaN,NaN,NaN
82,ThPh_Vent,Ventilation heat loss at thermal power plants.,Conversion,Heat loss,NaN,heat,NaN,NaN,NaN
83,wet_biomass_anaerobic_digestion_gas_separation,Wet biomass anaerobic digestion with gas separ...,Conversion,Digestion,Methane,BioWet,"CH4, CO2",NaN,NaN


In [7]:
# Technology Attributes
map_tech = {    
    'tech_name': ('Technology ID', 'tech_id'),
    'main_operation_unit': ('Technology ID', 'unit_operation'),
    'technology_description': ('Technology ID', 'description'),
    'technology_variant': ('Technology ID', 'category_spec'),
}

# Process Attributes
map_process = {    
    'process_name': ('Process ID', 'Process ID'),
    'process_description': ('Process ID', 'Process Description'),
    'process_type': ('Process ID', 'Tech Type'),          # Note: Old tech_type -> New process_type
    'process_category': ('Process ID', 'Process Category'),
    'feedstocks': ('Process ID', 'Feedstocks'),
    'products': ('Process ID', 'Products'),
    
    'main_sector': ('Technology ID', 'main_sector')
}

In [24]:
# Build mapped Technology and Process dataframes from existing mapping dictionaries

source_frames = {
    "Technology ID": df_tech_id,
    "Process ID": df_proc_id,
}

def build_mapped_df(mapping, source_frames):
    out = pd.DataFrame()
    for target_col, (sheet_name, source_col) in mapping.items():
        df_src = source_frames[sheet_name]
        out[target_col] = df_src[source_col] if source_col in df_src.columns else pd.NA
    return out

# 1) Technology dataframe
df_technology = build_mapped_df(map_tech, source_frames)

# 2) Process dataframe (base fields from Process ID + main_sector from Technology ID)
df_process = build_mapped_df(
    {k: v for k, v in map_process.items() if v[0] == "Process ID"},
    source_frames
)

# Derive process -> main_sector lookup from Technology ID sheet
process_sector_lookup = (
    df_tech_id[["process_id", "main_sector"]]
    .dropna(subset=["process_id"])
    .drop_duplicates(subset=["process_id"])
    .set_index("process_id")["main_sector"]
)

df_process["main_sector"] = df_process["process_name"].map(process_sector_lookup)

# Optional cleanup
df_process = df_process.drop_duplicates().reset_index(drop=True)
df_technology = df_technology.drop_duplicates().reset_index(drop=True)

# add id at the first column with format (tech_00001)
df_technology.insert(0, "tech_id", df_technology.index.map(lambda x: f"tech_{x+1:05d}"))
df_process.insert(0, "process_id", df_process.index.map(lambda x: f"proc_{x+1:05d}"))

print("Technology dataframe shape:", df_technology.shape)
print("Process dataframe shape:", df_process.shape)

display(df_technology.head())
display(df_process.head())

Technology dataframe shape: (95, 5)
Process dataframe shape: (85, 8)


,tech_id,tech_name,main_operation_unit,technology_description,technology_variant
0,tech_00001,Ammonia_HaberBosch,Haber Bosch unit,Ammonia synthesis via Haber Bosch process.,Haber Bosch
1,tech_00002,Ammonia_OCGT,OCGT,,Ammonia fueled
2,tech_00003,Amine_Washing_CO2,Amine washing unit,CO2 post combustion capture via Amine washing....,Amine washing
3,tech_00004,Waste_Venting_CO2,Ventilator,CO2 venting from waste-related processes.,Waste
4,tech_00005,Wood_Venting_CO2,Ventilator,CO2 venting from wood-related processes.,Wood


,process_id,process_name,process_description,process_type,process_category,feedstocks,products,main_sector
0,proc_00001,Ammonia_HaberBosch,Synthesizing Ammonia from Haber Bosch Process,Conversion,Haber Bosch,NaN,NaN,Ammonia
1,proc_00002,biomass_gasification_to_syngas,Biomass gasification to syngas,Conversion,Gasification,"Wood, O2",H2,NaN
2,proc_00003,CarbonCapture_Amine,CO2 post combustion capture via Amine washing....,Conversion,Carbon Capture,"CO2, amine",NaN,Carbon Capture
3,proc_00004,CarbonCapture_CO2Waste_Vent,CO₂ venting from waste-related processes.,Conversion,Carbon Capture,waste,NaN,Carbon Capture
4,proc_00005,CarbonCapture_CO2Wood_Vent,CO₂ venting from wood-related processes.,Conversion,Carbon Capture,wood,NaN,Carbon Capture


In [27]:
# export to csv

if save_to_db:
    df_technology.to_csv(r"motel-db\secondary\technology.csv", index=False)
    print(f'Technology data saved to CSV')

    df_process.to_csv(r"motel-db\secondary\process.csv", index=False)
    print(f'Process data saved to CSV')


Technology data saved to CSV
Process data saved to CSV


## 2.2 Source (done)

In [41]:
# load source data from reFuel.ch data
df_ref = refuel_data["Reference ID"].copy()
df_ref = df_ref.loc[1:].reset_index(drop=True)

# convert to motel-db source format
access_date_iso = pd.to_datetime(df_ref["Access Date"], errors="coerce").dt.strftime("%Y-%m-%d")
access_date_iso = access_date_iso.where(pd.notna(access_date_iso), None)

records_source = [
    {
        "source_name": row["Reference ID"],
        "source_description": row["Reference Description"] if pd.notna(row["Reference Description"]) else None,
        "source_type": None,
        "link": (
            None
            if pd.isna(row["DOI or Link"]) or str(row["DOI or Link"]).strip().lower() in {"na", "nan", ""}
            else str(row["DOI or Link"]).strip()
        ),
        "access_date": access_date_iso.iloc[i],
        "confidence_level": None,
        "assessment_method": None,
        "assessment_date": None,
    }
    for i, row in df_ref.iterrows()
]

df_source = pd.DataFrame(records_source)

df_source.insert(0, "source_id", df_source.index.map(lambda x: f"src_{x+1:05d}"))

df_source

,source_id,source_name,source_description,source_type,link,access_date,confidence_level,assessment_method,assessment_date
0,src_00001,report_VSE_2022,"VSE Energiezukunft 2050, Studie 2022",None,https://www.strom.ch/de/energiezukunft-2050/do...,2023-09-06,None,None,None
1,src_00002,assumption_Empa_2022,Expert Assumption,None,NaN,NaN,None,None,None
2,src_00003,paper_mutschler_2018,NaN,None,https://doi.org/10.1016/j.jcat.2018.08.002,NaN,None,None,None
3,src_00004,assumption_reFuelWP3_2025,Expert Assumption,None,NaN,NaN,None,None,None
4,src_00005,report_VSE_2025,"VSE Energiezukunft 2050, Update 2025",None,https://www.strom.ch/de/energiezukunft-2050/do...,2025-05-26,None,None,None
5,src_00006,paper_müller_PowerX_2025,The costs of future energy technologies: A com...,None,https://doi.org/10.1016/j.jcou.2025.103019,NaN,None,None,None
6,src_00007,assumption_Empa_2025,Expert assumptions and calculations by Empa,None,NaN,NaN,None,None,None
7,src_00008,paper_alpinePV_2024,ETHZ paper on Alpine PV cost and financial via...,None,https://doi.org/10.1016/j.apenergy.2024.124019,NaN,None,None,None
8,src_00009,paper_PVcostProjection_2025,"review the cost projection of PV, wind, battery",None,https://doi.org/10.1016/j.apenergy.2025.125856,NaN,None,None,None
9,src_00010,assumption_PVcostProjection,4/3/2% cost reduction per year in 2030/2040/20...,None,NaN,NaN,None,None,None


In [ ]:
# export to csv

if save_to_db:
    df_source.to_csv(r"motel-db\secondary\source.csv", index=False)
    print(f'Source data saved to CSV')

Source data saved to CSV


## 2.3 Attribute (to be checked)

exclude attirbutes for balancing, deal with it for balancing later

In [58]:
## load the pre-defined mapping for attribute

path_attr_mapping = r"data\refuel2motel_map\attribute.csv"

df_attr_mapping = pd.read_csv(path_attr_mapping)
df_attr_mapping

,Raw Attribute,attribute_name,attribute_description,unit,data_format,attribute_note
0,overall_efficiency,efficiency_overall,Ratio of total useful output energy to total i...,-,Float,Standard efficiency metric.
1,"Input, Output for Overall Efficiency",efficiency_basis,List of input and output carriers used to calc...,-,List of Strings,Defines the carriers for the efficiency calc.
2,theoretical_efficiency,efficiency_theoretical,Maximum thermodynamic efficiency (Carnot limit).,-,Float,Benchmark for performance.
3,operating_temperature_c,temp_operating,Typical operating temperature of the process.,°C,Float,Critical Fix: Was mapped to discount rate in s...
4,min_installation_size,capacity_min,Minimum viable installation capacity.,kW or kg/h,Float,Critical Fix: Was mapped to Value Range Check.
5,lifetime_yr,lifetime_economic,Economic lifetime of the asset.,yr,Integer,Critical Fix: Was mapped to uncertainty rating.
6,capex_one_time_eur,capex_fixed,Fixed capital expenditure (independent of size).,CUR,Float,Critical Fix: Was mapped to LCA ID.
7,capex_power_capacity_eur_per_kw,capex_variable,Variable capital expenditure per unit of capac...,CUR/kW,Float,Critical Fix: Was mapped to LCA unit.
8,opex_one_time_eur,opex_fixed,Fixed operating expenditure (annual).,CUR/yr,Float,Critical Fix: Was mapped to Embedded Carbon.
9,opex_fix_pct_of_capex,opex_fixed_pct,Fixed OPEX expressed as % of Total CAPEX.,%,Float,Alternative definition for opex_fixed.


# 3) Create record from reFuel.ch data sheets

TODO: adept the conversion for balancing

In [59]:
## --- function to convert reFuel.ch data into 'unmapped record'

import math
import re


def is_nan(value):
    return value is None or (
        isinstance(value, float) and math.isnan(value)
    )


def clean(value):
    return None if is_nan(value) else value


def parse_number_with_unit(text):
    """
    Converts:
        '2800 CHF/kg/h'
    into:
        (2800.0, 'CHF/kg/h')
    """
    if not isinstance(text, str):
        return text, None

    match = re.match(r"^\s*([0-9.]+)\s*(.*)$", text)
    if match:
        return float(match.group(1)), match.group(2).strip()

    return text, None


def split_csv(value):
    if is_nan(value):
        return []
    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip() and x.strip().lower() not in {"na", "nan"}
    ]


def split_csv_float(value):
    out = []
    for token in split_csv(value):
        try:
            out.append(float(token))
        except ValueError:
            out.append(None)
    return out


def build_balancing_entries(carriers, shares, units):
    size = max(len(carriers), len(shares), len(units))
    entries = []

    for i in range(size):
        carrier = carriers[i] if i < len(carriers) else None
        share = shares[i] if i < len(shares) else None
        unit = units[i] if i < len(units) else None

        if carrier is None and share is None and unit is None:
            continue

        entries.append({
            "carrier": carrier,
            "share": share,
            "unit": unit,
        })

    return entries


def normalize_unit(unit_text):
    if is_nan(unit_text):
        return None
    unit = str(unit_text).strip()
    if not unit or unit == "-":
        return None
    return unit


def build_attribute_specs(mapping_df):
    required_cols = {"Raw Attribute", "attribute_name", "attribute_description", "unit"}
    if not isinstance(mapping_df, pd.DataFrame) or not required_cols.issubset(mapping_df.columns):
        return []

    specs = []
    for _, rec in mapping_df.iterrows():
        raw_attr = clean(rec.get("Raw Attribute"))
        attr_name = clean(rec.get("attribute_name"))

        if not raw_attr or not attr_name:
            continue

        specs.append({
            "raw_attribute": str(raw_attr),
            "attribute_name": str(attr_name),
            "attribute_description": clean(rec.get("attribute_description")),
            "unit": normalize_unit(rec.get("unit")),
        })

    # De-duplicate by target attribute name, keep first mapping row.
    deduped = {}
    for spec in specs:
        deduped.setdefault(spec["attribute_name"], spec)

    return list(deduped.values())


def get_mapped_raw_value(row, raw_attribute, input_carriers, output_carriers):
    # This row in mapping is conceptual; derive from parsed carriers.
    if raw_attribute == "Input, Output for Overall Efficiency":
        combined = input_carriers + output_carriers
        return combined if combined else None

    return clean(row.get(raw_attribute))


def add_value(values, name, description, value, unit=None,
              scenario=None, time_index=None):

    if is_nan(value):
        return

    if isinstance(value, bool):
        value_format = "boolean"
    elif isinstance(value, int):
        value_format = "int"
    elif isinstance(value, float):
        value_format = "float"
    elif isinstance(value, (list, tuple)):
        value_format = "array"
    elif isinstance(value, dict):
        value_format = "dict"
    else:
        value_format = "text"

    record = {
        "attribute_name": name,
        "attribute_description": description,
        "unit": unit,
        "value": value,
        "value_format": value_format
    }

    if scenario:
        record["scenario"] = scenario

    if time_index:
        record["time_index"] = time_index

    values.append(record)


def map_technology(row):

    result = {
        "technology_name": (
            row.get("tech_id", "")
            .replace("_", " ")
        ),

        "description": clean(row.get("description")),

        "technology": {
            "technology_type": clean(row.get("tech_type")),
            "technology_category": clean(row.get("main_category")),
            "technology_assumption": (
                f"{row.get('cost_base_year')} reference technology"
                if not is_nan(row.get("cost_base_year"))
                else None
            ),
            "process_name": clean(row.get("unit_operation")),
            "process_type": clean(row.get("process_type")),
            "process_category": clean(row.get("main_sector")),
            "process_assumption": clean(row.get("category_spec"))
        },

        "scope": {
            "geographic_scope_description":
                clean(row.get("Cost Base")),
            "temporal_scope_description":
                str(row.get("cost_base_year"))
                if not is_nan(row.get("cost_base_year"))
                else None,
            "capacity_scope_description":
                clean(row.get("min_installation_size")),
            "system_boundary_description":
                clean(row.get("tech_boundary")),
            "scope_assumption":
                clean(row.get("tech_maturity"))
        },

        "sources": [],
        "values": [],
        "balancing": {
            "input": [],
            "output": []
        },
        "tags": {
            "related_project": "eHubX",
            "tags": []
        }
    }

    values = result["values"]

    # -----------------------
    # Source parsing
    # -----------------------
    source_text = row.get("list_of_source_id")

    if isinstance(source_text, str):

        if "AECOM_2022" in source_text:
            result["sources"].append({
                "source_description": "AECOM_2022",
                "source_type": "report",
                "assessment_method": "literature"
            })

        if "parameters_lca" in source_text:
            result["sources"].append({
                "source_description": "parameters_lca",
                "source_type": "database",
                "assessment_method": "modeled"
            })

    # -----------------------
    # Inputs and outputs for balancing
    # -----------------------
    input_carriers = split_csv(row.get("carriers_in"))
    input_shares = split_csv_float(row.get("ratios_in"))
    input_units = split_csv(row.get("units_in"))

    output_carriers = split_csv(row.get("carriers_out"))
    output_shares = split_csv_float(row.get("ratios_out"))
    output_units = split_csv(row.get("units_out"))

    result["balancing"]["input"] = build_balancing_entries(
        input_carriers,
        input_shares,
        input_units,
    )
    result["balancing"]["output"] = build_balancing_entries(
        output_carriers,
        output_shares,
        output_units,
    )

    # -----------------------
    # Attributes from mapping CSV only
    # -----------------------
    specs = build_attribute_specs(df_attr_mapping)
    for spec in specs:
        value = get_mapped_raw_value(
            row,
            spec["raw_attribute"],
            input_carriers,
            output_carriers,
        )
        add_value(
            values,
            spec["attribute_name"],
            spec["attribute_description"],
            value,
            unit=spec["unit"],
        )

    # -----------------------
    # Tags
    # -----------------------
    tags = result["tags"]["tags"]

    for field in [
        "main_sector",
        "main_category",
        "category_spec",
        "tech_type",
        "tech_maturity"
    ]:
        value = clean(row.get(field))
        if value:
            tags.append(str(value))

    return result

In [60]:
## run maping

sheet_name = 'ConvTech'

# read row by row (per technology)
df_conv = refuel_data[sheet_name]
df_conv.columns = df_conv.loc[2]
df_conv = df_conv.loc[3:].reset_index(drop=True)

df_conv
for indx in df_conv.index:
    row = df_conv.loc[indx]    

    record = map_technology(row)
    print(f"Record for technology '{record['technology_name']}':")

    break

record

Record for technology 'Amine Washing CO2 2030':


{'technology_name': 'Amine Washing CO2 2030',
 'description': 'CO2 post combustion capture via Amine washing. Temperature swing process. 2030 reference based on an approx. 150 ktCO2/a plant size. ',
 'technology': {'technology_type': 'Conversion',
  'technology_category': 'Point source capture',
  'technology_assumption': '2030 reference technology',
  'process_name': 'Amine Absorber',
  'process_type': 'CO2 capture',
  'process_category': 'CO2 capture',
  'process_assumption': 'Amine washing'},
 'scope': {'geographic_scope_description': 'CH',
  'temporal_scope_description': '2030',
  'capacity_scope_description': '20000 kg/h',
  'system_boundary_description': 'Plant ready to operate',
  'scope_assumption': 'Mature'},
 'sources': [{'source_description': 'AECOM_2022',
   'source_type': 'report',
   'assessment_method': 'literature'},
  {'source_description': 'parameters_lca',
   'source_type': 'database',
   'assessment_method': 'modeled'}],
 'values': [{'attribute_name': 'efficiency_ba

In [61]:
# quick check: new balancing payload
record.get("balancing")

{'input': [{'carrier': 'CO2Fluegas', 'share': 1.0, 'unit': 'kg'},
  {'carrier': 'El13', 'share': 0.303, 'unit': 'kWh'},
  {'carrier': 'ThHt', 'share': 0.71, 'unit': 'kWh'}],
 'output': [{'carrier': 'CO2Sys', 'share': 0.95, 'unit': 'kg'}]}

In [62]:
# quick check: attributes come only from mapping CSV
mapped_attr_names = set(df_attr_mapping["attribute_name"].dropna().astype(str).str.strip())
record_attr_names = {v["attribute_name"] for v in record.get("values", [])}

print("record attributes:", len(record_attr_names))
print("not in mapping:", sorted(record_attr_names - mapped_attr_names))

record attributes: 12
not in mapping: []


In [ ]:
#TODO the values here is not right!

In [63]:
record.get('values')

[{'attribute_name': 'efficiency_basis',
  'attribute_description': 'List of input and output carriers used to calculate overall efficiency.',
  'unit': None,
  'value': ['CO2Fluegas', 'El13', 'ThHt', 'CO2Sys'],
  'value_format': 'array'},
 {'attribute_name': 'efficiency_theoretical',
  'attribute_description': 'Maximum thermodynamic efficiency (Carnot limit).',
  'unit': None,
  'value': 'na',
  'value_format': 'text'},
 {'attribute_name': 'capacity_min',
  'attribute_description': 'Minimum viable installation capacity.',
  'unit': 'kW or kg/h',
  'value': '20000 kg/h',
  'value_format': 'text'},
 {'attribute_name': 'lifetime_economic',
  'attribute_description': 'Economic lifetime of the asset.',
  'unit': 'yr',
  'value': 25,
  'value_format': 'int'},
 {'attribute_name': 'capex_fixed',
  'attribute_description': 'Fixed capital expenditure (independent of size).',
  'unit': 'CUR',
  'value': 0,
  'value_format': 'int'},
 {'attribute_name': 'capex_variable',
  'attribute_description': 